# TP 3 : Ingenierie Deep Learning — TensorFlow vs PyTorch

**Session 2026 — D1**

Ce notebook compare les deux ecosystemes majeurs du Deep Learning (PyTorch et
TensorFlow) a travers six exercices : algebre tensorielle, differenciation
automatique, pipeline de classification d'images (CNN) et gestion multi-device.

La graine aleatoire est fixee a `42` pour garantir la reproductibilite des
resultats d'une execution a l'autre.

## Introduction : configuration de l'environnement

Verification des versions installees et des ressources materielles disponibles
(GPU CUDA/NVIDIA ou Apple Silicon).

In [1]:
import os
import random

import numpy as np
import torch
import tensorflow as tf

print(f"NumPy Version      : {np.__version__}")
print(f"PyTorch Version    : {torch.__version__} | CUDA Available : {torch.cuda.is_available()}")
print(f"TensorFlow Version : {tf.__version__} | GPU Detected : {tf.config.list_physical_devices('GPU')}")

C:\Users\FatimetouELALEM\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


NumPy Version      : 2.1.3
PyTorch Version    : 2.10.0+cpu | CUDA Available : False
TensorFlow Version : 2.19.1 | GPU Detected : []


### Reproductibilite

On centralise la fixation de la graine aleatoire : chaque bibliotheque possede
son propre generateur, il faut donc les initialiser separement.

In [2]:
SEED = 42


def fixer_graine(graine: int = SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(graine)
    random.seed(graine)
    np.random.seed(graine)
    torch.manual_seed(graine)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(graine)
    tf.random.set_seed(graine)


def obtenir_peripherique_torch():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


fixer_graine(SEED)

## Partie 1 : Fondations et algebre tensorielle

### Exercice 1 : Initialisation comparee

Creation en parallele des memes structures sous PyTorch et TensorFlow : tenseur
de zeros `3x4`, matrice identite `5x5`, tenseur aleatoire `N(0,1)` de taille
`2x3x3`. La graine est fixee de facon identique des deux cotes.

In [3]:
fixer_graine(SEED)

# --- PyTorch ---
zeros_t = torch.zeros(3, 4)
identite_t = torch.eye(5)
aleatoire_t = torch.randn(2, 3, 3)

# --- TensorFlow ---
zeros_f = tf.zeros((3, 4))
identite_f = tf.eye(5)
aleatoire_f = tf.random.normal((2, 3, 3))

print("=== PyTorch ===")
print("Zeros (3x4) :\n", zeros_t)
print("Identite (5x5) :\n", identite_t)
print("Aleatoire N(0,1) (2x3x3) :\n", aleatoire_t)

print("\n=== TensorFlow ===")
print("Zeros (3x4) :\n", zeros_f.numpy())
print("Identite (5x5) :\n", identite_f.numpy())
print("Aleatoire N(0,1) (2x3x3) :\n", aleatoire_f.numpy())

=== PyTorch ===
Zeros (3x4) :
 tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
Identite (5x5) :
 tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]])
Aleatoire N(0,1) (2x3x3) :
 tensor([[[ 1.9269,  1.4873, -0.4974],
         [ 0.4396, -0.7581,  1.0783],
         [ 0.8008,  1.6806,  0.3559]],

        [[-0.6866,  0.6105,  1.3347],
         [-0.2316,  0.0418, -0.2516],
         [ 0.8599, -0.3097, -0.3957]]])

=== TensorFlow ===
Zeros (3x4) :
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Identite (5x5) :
 [[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]
Aleatoire N(0,1) (2x3x3) :
 [[[ 0.3274685  -0.8426258   0.3194337 ]
  [-1.4075519  -2.3880599  -1.0392479 ]
  [-0.5573232   0.539707    1.6994323 ]]

 [[ 0.28893656 -1.5066116  -0.26454744]
  [-0.59722406 -1.9171132  -0.62044144]
  [ 0.8504023  -0.40604794 -3.0258412 ]]]

### Exercice 2 : Typage, interoperabilite NumPy et reshaping

On part du tableau `np.arange(1, 13).reshape(3, 4)` et on etudie la conversion
en `float32`, le partage de memoire (PyTorch) contre la copie (TensorFlow),
puis le redimensionnement `2x6` et l'aplatissement en 1D.

In [4]:
arr = np.arange(1, 13).reshape(3, 4)

# torch.from_numpy partage le buffer memoire ; tf.convert_to_tensor copie.
tenseur_torch = torch.from_numpy(arr).to(torch.float32)
tenseur_tf = tf.convert_to_tensor(arr, dtype=tf.float32)

print("Tableau NumPy initial :\n", arr)
print("Type PyTorch    :", tenseur_torch.dtype)
print("Type TensorFlow :", tenseur_tf.dtype)

Tableau NumPy initial :
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
Type PyTorch    : torch.float32
Type TensorFlow : <dtype: 'float32'>


In [5]:
# Partage de memoire vs copie : on garde le type entier natif pour observer
# le phenomene (la conversion float32 ci-dessus cree un nouveau buffer).
arr_test = np.arange(1, 13).reshape(3, 4)
vue_torch = torch.from_numpy(arr_test)
copie_tf = tf.convert_to_tensor(arr_test)

arr_test[0, 0] = 999

print("Apres modification de arr[0, 0] = 999 :")
print("  NumPy       arr[0,0]     =", arr_test[0, 0])
print("  PyTorch     tenseur[0,0] =", vue_torch[0, 0].item(), "(memoire partagee -> valeur modifiee)")
print("  TensorFlow  tenseur[0,0] =", int(copie_tf[0, 0].numpy()), "(copie -> valeur inchangee)")

Apres modification de arr[0, 0] = 999 :
  NumPy       arr[0,0]     = 999
  PyTorch     tenseur[0,0] = 999 (memoire partagee -> valeur modifiee)
  TensorFlow  tenseur[0,0] = 1 (copie -> valeur inchangee)


In [6]:
# Reshaping (2x6) puis flatten (1D).
torch_2x6 = tenseur_torch.view(2, 6)   # view() exige un tenseur contigu
torch_1d = torch_2x6.flatten()

tf_2x6 = tf.reshape(tenseur_tf, (2, 6))
tf_1d = tf.reshape(tf_2x6, (-1,))      # -1 aplatit en 1D

print("PyTorch 2x6 :\n", torch_2x6)
print("PyTorch 1D  :", torch_1d)
print("TensorFlow 2x6 :\n", tf_2x6.numpy())
print("TensorFlow 1D  :", tf_1d.numpy())

PyTorch 2x6 :
 tensor([[ 1.,  2.,  3.,  4.,  5.,  6.],
        [ 7.,  8.,  9., 10., 11., 12.]])
PyTorch 1D  : tensor([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12.])
TensorFlow 2x6 :
 [[ 1.  2.  3.  4.  5.  6.]
 [ 7.  8.  9. 10. 11. 12.]]
TensorFlow 1D  : [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12.]


## Partie 2 : Graphes de calcul et differenciation automatique

### Exercice 3 : Descente de gradient sur la fonction de Rosenbrock

$$f(x, y) = (a - x)^2 + b\,(y - x^2)^2 \quad\text{avec } a = 1,\ b = 100.$$

Minimum global en `(1, 1)`. Depart en `(0, 0)`, 100 iterations, pas
`eta = 0.002`. PyTorch utilise `requires_grad=True` + `backward()`, TensorFlow
le gestionnaire de contexte `tf.GradientTape()`.

In [7]:
A, B = 1.0, 100.0
POINT_INITIAL = (0.0, 0.0)
NB_ITERATIONS = 100
ETA = 0.002


def rosenbrock(x, y):
    return (A - x) ** 2 + B * (y - x ** 2) ** 2


# --- PyTorch : autograd ---
x = torch.tensor(POINT_INITIAL[0], requires_grad=True)
y = torch.tensor(POINT_INITIAL[1], requires_grad=True)
for _ in range(NB_ITERATIONS):
    perte = rosenbrock(x, y)
    perte.backward()                     # calcule df/dx et df/dy
    with torch.no_grad():                # mise a jour hors du graphe autograd
        x -= ETA * x.grad
        y -= ETA * y.grad
        x.grad.zero_()
        y.grad.zero_()
res_torch = (x.item(), y.item(), rosenbrock(x, y).item())

# --- TensorFlow : GradientTape ---
xt = tf.Variable(POINT_INITIAL[0])
yt = tf.Variable(POINT_INITIAL[1])
for _ in range(NB_ITERATIONS):
    with tf.GradientTape() as tape:
        perte = rosenbrock(xt, yt)
    grad_x, grad_y = tape.gradient(perte, [xt, yt])
    xt.assign_sub(ETA * grad_x)
    yt.assign_sub(ETA * grad_y)
res_tf = (float(xt.numpy()), float(yt.numpy()), float(rosenbrock(xt, yt).numpy()))

print(f"Point initial : {POINT_INITIAL}, eta = {ETA}, iterations = {NB_ITERATIONS}")
print(f"PyTorch    -> (x, y) = ({res_torch[0]:.6f}, {res_torch[1]:.6f}) | f = {res_torch[2]:.6e}")
print(f"TensorFlow -> (x, y) = ({res_tf[0]:.6f}, {res_tf[1]:.6f}) | f = {res_tf[2]:.6e}")
print("Minimum theorique : (1.0, 1.0) avec f = 0.0")

Point initial : (0.0, 0.0), eta = 0.002, iterations = 100
PyTorch    -> (x, y) = (0.299179, 0.086415) | f = 4.921065e-01
TensorFlow -> (x, y) = (0.299179, 0.086415) | f = 4.921065e-01
Minimum theorique : (1.0, 1.0) avec f = 0.0


## Partie 3 : Pipeline de classification d'images (CNN)

### Exercice 4 : Implementation du modele

Architecture (entree `1 x 28 x 28`) : `Conv2D(32, 3x3, ReLU)` -> `MaxPool(2x2)`
-> `Flatten` -> `Dense(128, ReLU)` -> `Dense(10, logits)`.

Apres la convolution 3x3 sans padding, la carte passe de 28x28 a 26x26, puis le
pooling 2x2 la reduit a 13x13. La couche dense recoit donc `13*13*32 = 5408`
entrees.

| Composant | PyTorch (`nn.Module`) | TensorFlow (`tf.keras`) |
|---|---|---|
| Couche convolutive | `nn.Conv2d(in_channels, ...)` | `tf.keras.layers.Conv2D(...)` |
| Mode entrainement | `model.train()` | argument `training=True` |
| Calcul de la loss | `nn.CrossEntropyLoss()` | `SparseCategoricalCrossentropy(from_logits=True)` |

In [8]:
import torch.nn as nn


class CNNPyTorch(nn.Module):
    def __init__(self, nb_classes: int = 10):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(13 * 13 * 32, 128)
        self.fc2 = nn.Linear(128, nb_classes)

    def forward(self, x):
        x = self.relu(self.conv(x))
        x = self.pool(x)
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x)   # logits (pas de softmax : gere par la loss)


def construire_cnn_tensorflow(nb_classes: int = 10) -> tf.keras.Model:
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(32, kernel_size=3, activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(nb_classes),
    ])


modele_torch = CNNPyTorch()
print("=== Modele PyTorch ===")
print(modele_torch)
print("Nombre de parametres :", sum(p.numel() for p in modele_torch.parameters()))

modele_tf = construire_cnn_tensorflow()
print("\n=== Modele TensorFlow ===")
modele_tf.summary()

=== Modele PyTorch ===
CNNPyTorch(
  (conv): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=5408, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)
Nombre de parametres : 693962



=== Modele TensorFlow ===


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 5408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       692,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 693,962 (2.65 MB)

 Trainable params: 693,962 (2.65 MB)

 Non-trainable params: 0 (0.00 B)

### Exercice 5 : Boucle d'entrainement personnalisee

Entrainement du CNN sur une epoque de FashionMNIST, sans `model.fit()`. Les cinq
etapes sont codees explicitement : forward, loss, zero-grad, backward et mise a
jour via Adam (`eta = 0.001`). Le jeu de donnees est charge une seule fois via
`tf.keras.datasets` puis partage entre les deux frameworks (mini-batches
identiques).

In [9]:
TAILLE_BATCH = 64
TAUX_APPRENTISSAGE = 0.001
NB_BATCHES_DEMO = 100   # sous-ensemble pour une execution rapide sur CPU

(x_train, y_train), _ = tf.keras.datasets.fashion_mnist.load_data()
x_train = (x_train.astype("float32") / 255.0)[..., np.newaxis]   # (N, 28, 28, 1)
y_train = y_train.astype("int64")


def iterer_batches(x, y, taille_batch, nb_batches):
    for i in range(nb_batches):
        debut, fin = i * taille_batch, (i + 1) * taille_batch
        if fin > len(x):
            break
        yield x[debut:fin], y[debut:fin]

In [10]:
# --- Boucle PyTorch ---
fixer_graine(SEED)
peripherique = obtenir_peripherique_torch()
modele = CNNPyTorch().to(peripherique)
critere = nn.CrossEntropyLoss()
optimiseur = torch.optim.Adam(modele.parameters(), lr=TAUX_APPRENTISSAGE)

modele.train()
perte_cumulee, nb = 0.0, 0
for images, labels in iterer_batches(x_train, y_train, TAILLE_BATCH, NB_BATCHES_DEMO):
    images = torch.from_numpy(images).permute(0, 3, 1, 2).to(peripherique)  # (N, C, H, W)
    labels = torch.from_numpy(labels).to(peripherique)

    logits = modele(images)          # 1. forward
    perte = critere(logits, labels)  # 2. loss
    optimiseur.zero_grad()           # 3. zero-grad
    perte.backward()                 # 4. backward
    optimiseur.step()                # 5. mise a jour

    perte_cumulee += perte.item()
    nb += 1

print(f"PyTorch    : perte moyenne = {perte_cumulee / nb:.4f}")

PyTorch    : perte moyenne = 0.8740


In [11]:
# --- Boucle TensorFlow ---
fixer_graine(SEED)
modele_k = construire_cnn_tensorflow()
critere_k = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimiseur_k = tf.keras.optimizers.Adam(learning_rate=TAUX_APPRENTISSAGE)

perte_cumulee, nb = 0.0, 0
for images, labels in iterer_batches(x_train, y_train, TAILLE_BATCH, NB_BATCHES_DEMO):
    images = tf.convert_to_tensor(images)   # (N, H, W, C) natif Keras
    labels = tf.convert_to_tensor(labels)

    with tf.GradientTape() as tape:
        logits = modele_k(images, training=True)  # 1. forward
        perte = critere_k(labels, logits)         # 2. loss
    gradients = tape.gradient(perte, modele_k.trainable_variables)   # 3. + 4.
    optimiseur_k.apply_gradients(zip(gradients, modele_k.trainable_variables))  # 5.

    perte_cumulee += float(perte.numpy())
    nb += 1

print(f"TensorFlow : perte moyenne = {perte_cumulee / nb:.4f}")

TensorFlow : perte moyenne = 0.7531


## Partie 4 : Gestion avancee du materiel et profiling

### Exercice 6 : Migration multi-device (CPU <-> GPU)

Les transferts repetes entre memoire hote (RAM) et memoire device (VRAM) sont un
goulot d'etranglement classique. On place explicitement modele et donnees sur le
meilleur peripherique disponible dans chaque framework.

In [12]:
# --- PyTorch : detection cuda > mps > cpu ---
peripherique = obtenir_peripherique_torch()
print("PyTorch - peripherique retenu :", peripherique)

modele_lin = nn.Linear(10, 2).to(peripherique)
mini_batch = torch.randn(64, 10, device=peripherique)
sortie = modele_lin(mini_batch)
print("  Sortie calculee sur :", sortie.device)
print("  Apres .cpu() :", sortie.detach().cpu().device)   # rapatriement en RAM

PyTorch - peripherique retenu : cpu
  Sortie calculee sur : cpu
  Apres .cpu() : cpu


In [13]:
# --- TensorFlow : placement implicite + forcage via tf.device ---
gpus = tf.config.list_physical_devices("GPU")
cible = "/GPU:0" if gpus else "/CPU:0"
print("TensorFlow - placement force sur :", cible)

with tf.device(cible):
    matrice = tf.random.normal((64, 10))
    resultat = tf.matmul(matrice, tf.random.normal((10, 2)))
print("  Operation placee sur :", resultat.device)

TensorFlow - placement force sur : /CPU:0
  Operation placee sur : /job:localhost/replica:0/task:0/device:CPU:0


### Question de reflexion

**PyTorch** : un tenseur situe sur le GPU pointe vers de la memoire VRAM, alors
que NumPy ne manipule que de la memoire hote (RAM). Appeler `.numpy()`
directement sur un tenseur CUDA leve une erreur
(*can't convert cuda device type tensor to numpy*). Il faut d'abord rapatrier
les donnees avec `.cpu()` (et `.detach()` si le tenseur suit un gradient) avant
`.numpy()`.

**TensorFlow** : la conversion `.numpy()` gere ce rapatriement de maniere
implicite ; lorsqu'un tenseur reside sur le GPU, TensorFlow effectue
automatiquement la copie device -> hote lors de l'appel, sans etape manuelle
equivalente a `.cpu()`. La contrepartie est que cette copie silencieuse peut
masquer des transferts couteux si elle est appelee dans une boucle.